# 6 · Preference — what the training signal pushes toward  `[TRAINING]`

Inside the training signal: what, in words, the update pushes the policy toward, how that target moves across iterations, and **whether it predicts the eval move it produced**. Exports → `results/<VIEW>/figures|tables/6_preference/`.

**§1–§2 — PTO's preference pairs** (the original Mass-Mean-Probe over `pref_pairs/pairs.csv`): per iteration the unit **preference direction** = normalized mean(chosen − rejected) embedding, and projecting words / MI-concepts onto it reads out the preference.

**§3 — both methods, one probe (added 2026-08-02).** GRPO has no preference pairs, but "preference" was never the essential thing: both methods weight the candidates of a group and push the policy along the weighted sum, differing only in the weights — DPO puts **±1** on the recorded chosen/rejected, GRPO uses the **standardized group-relative advantage** that actually scales each completion's gradient. Rescaled to a common per-group size they are directly comparable, so *"what does each method actually reward?"* — the thesis's central question, previously unanswerable on the GRPO side — gets a like-for-like answer.

**§4 — does the signal predict the outcome?** Joins each iteration's update features to the persona-paired eval delta that iteration produced (`model_iter_n` vs `model_iter_{n-1}`).

> ⚠️ **§3 also re-measures the probe itself, and the news is not good for §1–§2.** A direction estimated from one iteration's PTO pairs has a **split-half cosine of ~0.19** — two halves of the same iteration point almost independently — and its held-out win rate is **0.55**, against the **0.68** in-sample number §1 reports. Read §1's per-iteration drift artifacts (word drift, learn/unlearn, MI-concept curves, direction drift) as **mostly estimation noise**; the claims that survive are the ones §3 makes from *pooled* directions, whose reliability is measured and reported.

In [ ]:
import sys, os
_p = os.path.abspath(".")                      # find eda/ (the dir holding eda_analysis/) from any depth
while _p != os.path.dirname(_p) and not os.path.isdir(os.path.join(_p, "eda_analysis")):
    _p = os.path.dirname(_p)
sys.path.insert(0, _p)
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
pd.set_option("display.width", 185, "display.max_columns", 50)
import eda_analysis
from eda_analysis import training, pref

# ╔═══ VIEW — the one knob ════════════════════════════════════════════════════════╗
# "all" = every arm | "L0" = K=0 arms (PTO_LA0/GRPO_LA0) | "L5" = K=5 arms (thin, paused).
# Sets BOTH the arm filter AND the results root -> results/<VIEW>/figures|tables/<group>/.
# Edit the default for interactive use; render_views.py overrides it via the EDA_VIEW env var.
# Sections 1-2 are PTO-only (they read pref_pairs/pairs.csv); sections 3-4 cover BOTH methods via
# the update-weighted candidate view, so the view's GRPO arms are analysed there.
VIEW = os.environ.get("EDA_VIEW", "L0")
# ╔═══ JUDGE (training-side notebook) ═════════════════════════════════════════════╗
# This notebook's sections read the TRAINING side (generations.jsonl candidate rewards, preference
# pairs, TensorBoard curves), which were produced by the training oracle during the run and CANNOT
# be re-graded after the fact. Re-rendering under a second judge would emit byte-identical figures
# into that judge's folder, implying a measurement that never happened -- so a non-primary JUDGE is
# refused here rather than silently honoured. Section 4 joins to the EVAL side, which IS
# judge-swappable in principle; it uses the primary oracle only, and says so. (The genuinely
# multi-judge work lives in 8_Measurement_Validity, which reads data/eval_scores/ directly.)
JUDGE = os.environ.get("EDA_JUDGE", "")
if JUDGE:
    raise SystemExit(f"{__name__}: EDA_JUDGE={JUDGE!r} refused - this notebook is training-side "
                     "and not judge-swappable; see 8_Measurement_Validity for the multi-judge analysis.")

cfg = eda_analysis.EdaConfig(
    view=VIEW,                             # arm filter + results/<VIEW>/ root
    export_group="6_preference",           # topic family = this notebook's number
    selection="all",
    focus_arms=None, focus_metric="Q1Q2",
)
S = eda_analysis.notebook_setup(cfg)
FOCUS = cfg.focus_arms or sorted(S.SCORES.arm.unique())

## 0 · Arms with training data
PTO arms with preference pairs drive §1–§2; §3–§4 pick up every arm of both methods from `generations.jsonl`.

In [ ]:
PTO_ARMS = [a for a in S.ARMS if a.method == "PTO"
            and (cfg.focus_arms is None or a.label in cfg.focus_arms)
            and len(training.load_pref_pairs([a]))]
print("PTO arms with preference pairs (sections 1-2):", [a.label for a in PTO_ARMS])
print("all arms in this view (sections 3-4):", [a.label for a in S.ARMS])

## 1 · Per-arm preference, iteration by iteration  `[TRAINING]`
**Purpose.** Per PTO arm: probe quality (does the direction separate the pairs? wins > 0.5) → pooled
word ranking → per-iteration word drift (heatmap + top-words table) → **direction drift in 2D** +
consecutive cosine → **learned vs unlearned words** → MI-concept drift + a first→last read-out.

> ⚠️ **`wins_correct` here is IN-SAMPLE** — the direction is scored on the very pairs it was fitted on, and at PTO's ~400 pairs an iteration that inflates it by ~0.13 (measured: 0.68 in-sample vs 0.55 held out). §3 reports both, plus the split-half cosine that says these per-iteration directions are only ~0.19 reliable. Everything in this section is therefore **more noise than it looks**; the pooled, audited versions are in §3.

In [ ]:
RESULTS = {}   # arm -> {DIRS, CAT} for the K0-vs-K5 comparison in §2

def analyze_pref(arm):
    PAIRS = pref.add_text_features(training.load_pref_pairs([arm]))
    if PAIRS.empty:
        print(f"{arm.label}: no pairs."); return
    EMB = pref.embed_pairs(PAIRS)
    DIRS = pref.preference_direction_by_iter(EMB)
    print(f"\n################  {arm.label}  ################")
    PQ = pref.probe_quality_by_iter(EMB, DIRS)
    print("[probe] wins_correct should be > 0.5 for a real preference axis:"); display(PQ.round(4))
    eda_analysis.save_table(PQ.round(4), f"{arm.label}_pref_probe_quality", caption=f"{arm.label} preference-probe quality per iteration (wins_correct, gap, margin).")
    words, wmat = pref.embed_vocab(pref.build_vocab(PAIRS, top_n=3000)); WP = pref.word_projection(words, wmat, DIRS)
    # overall preference + per-iteration word drift
    fig = pref.pref_word_ranking(WP, title=f"{arm.label}: words by preference projection (green=chosen, red=rejected)")
    if fig: eda_analysis.save_fig(fig, f"{arm.label}_pref_word_ranking", caption=f"{arm.label} top chosen/rejected-aligned words (Mass Mean Probe, pooled over iters)."); plt.show()
    fig = pref.pref_word_drift_heatmap(WP, title=f"{arm.label}: preferred-word drift across iterations")
    if fig: eda_analysis.save_fig(fig, f"{arm.label}_pref_word_drift", caption=f"{arm.label} per-iteration projection of the top chosen-/rejected-aligned words (drift)."); plt.show()
    display(pref.top_words_by_iter(WP, k=8))
    # direction drift (vectors) + learned/unlearned words
    fig = pref.plot_direction_drift(pref.preference_direction_drift(DIRS), title=f"{arm.label}: preference-direction drift")
    if fig: eda_analysis.save_fig(fig, f"{arm.label}_pref_direction_drift", caption=f"{arm.label} preference direction in 2D PCA + consecutive cosine (how the preferred axis re-orients across iterations)."); plt.show()
    fig = pref.plot_learn_unlearn(pref.learn_unlearn_words(WP, k=8))
    if fig: eda_analysis.save_fig(fig, f"{arm.label}_pref_learn_unlearn", caption=f"{arm.label} words most newly preferred (learned) vs dropped (unlearned) across iteration transitions."); plt.show()
    # MI-concept drift + first->last read-out
    CAT = pref.category_projection(DIRS)
    if not CAT.empty:
        fig = pref.plot_category_drift(CAT)
        if fig: eda_analysis.save_fig(fig, f"{arm.label}_pref_category_drift", caption=f"{arm.label} MI-concept word groups projected onto the chosen-rejected direction per iteration."); plt.show()
        eda_analysis.save_table(CAT.round(4), f"{arm.label}_pref_MI_concepts", caption=f"{arm.label} MI-concept projection onto the preference direction per iteration.")
        wide = CAT.pivot_table(index="category", columns="train_iter", values="score")
        f0, fl = wide.columns.min(), wide.columns.max()
        print(f"[MI-concept shift iter {f0}->{fl}; + = MORE preferred over training]:")
        for cat, d in (wide[fl] - wide[f0]).sort_values(ascending=False).items():
            print(f"   {cat:<16} {wide.loc[cat, f0]:+.3f} -> {wide.loc[cat, fl]:+.3f}   (Δ {d:+.3f})")
    RESULTS[arm.label] = {"DIRS": DIRS, "CAT": CAT}

for arm in PTO_ARMS:
    analyze_pref(arm)
if not PTO_ARMS:
    print("No PTO arm with preference pairs scored yet.")

## 2 · Does look-ahead change *what* is preferred? K=0 vs K=5  `[TRAINING]`

> ⚠️ **DESCRIPTIVE only — PTO_LA5 is thin (4 iters).** This K=0-vs-K=5 overlay is **hypothesis-generating**, not an inferential K test; read a divergence as a direction to confirm once the K=5 arm finishes, not as a tested effect. (Same caveat as `5_Training_and_Reliability` §4 / `7_Stats` §4.)

**Purpose.** Overlay PTO_LA0 vs PTO_LA5 MI-concept preference across iterations, and the cosine between
their per-iteration directions. **Read:** diverging curves / low cosine = look-ahead *suggests* steering the
preference toward different language.

In [ ]:
if {"PTO_LA0", "PTO_LA5"} <= set(RESULTS):
    fig = pref.plot_category_compare({a: RESULTS[a]["CAT"] for a in ("PTO_LA0", "PTO_LA5")},
                                     palette=S.PALETTE, title="MI-concept preference: K=0 vs K=5 (PTO)")
    if fig:
        eda_analysis.save_fig(fig, "pref_category_K0_vs_K5", caption="PTO MI-concept preference projection across iterations, K=0 vs K=5 — does look-ahead change what the policy prefers? Per-iteration directions, so see the reliability caveat in the header and the measured split-half in §3."); plt.show()
    d0, d5 = RESULTS["PTO_LA0"]["DIRS"], RESULTS["PTO_LA5"]["DIRS"]
    common = sorted(set(d0) & set(d5))
    if common:
        print("cos(dir_K0, dir_K5) at matched iters:", {i: round(float(d0[i] @ d5[i]), 3) for i in common})
        print("  ^ uncorrected for estimation noise — §3 reports the attenuation-corrected version.")
else:
    print("Need both PTO_LA0 and PTO_LA5 with pref pairs for the K comparison "
          f"(have: {sorted(RESULTS)}) — the cross-K read lives in the `all` render only.")

## 3 · One probe, both methods — what does each update actually reward?  `[TRAINING]`

**Purpose.** Put PTO and GRPO on the same axes. Every candidate carries the weight its method's update gives it (DPO's ±1 chosen/rejected; GRPO's standardized advantage), rescaled per group to a common size, so a "unit of push" means the same thing on both sides. Two complementary readouts:

- **Lexical push** (`weighted_lexical_contrast`) — exact, no embeddings, **every** group: Σ w·feature per group, so `+40` on length = "the update pushes toward ~40-character-longer completions" and `0` = indifferent. Comes with a standard error, because these are small per-pair numbers.
- **Semantic direction** (`direction_by_iter` / `direction_by_arm`) — `normalize(Σ w · embedding)`, the generalization of §1's Mass-Mean-Probe to any weighting. Everything downstream (word projection, MI concepts) takes a plain direction, so it works unchanged for GRPO.

**And the probe gets audited.** `direction_quality` reports three numbers §1 never had: `wins_holdout` (each half judged by the *other* half's direction — the honest version of §1's in-sample `wins_correct`), `split_half_cos` (is the direction even estimated?), and, for cross-arm cosines, the **attenuation ceiling** — two noisy directions cannot correlate to 1 even if identical, so a raw cosine means nothing without it. Same correction `8_Measurement_Validity` applies to cross-judge agreement, for the same reason.

**Sampling.** Directions cap at 400 groups per (arm, iteration) — chosen by measuring reliability at 50/100/200/400, not guessed. The lexical half uses everything.

In [ ]:
CANDS = pref.load_weighted_candidates(S.ARMS)
print("arms with weighted training candidates:", sorted(CANDS.arm.unique()) if not CANDS.empty else "none")

# ── 3a · the exact, embedding-free half: what the update pushes toward, per iteration ──
LEX = pref.weighted_lexical_contrast(CANDS)
display(LEX.round(4))
eda_analysis.save_table(LEX.round(4), "update_lexical_push", caption="Per (arm, iteration): the lexical contrast the update pushes for, Sum(w*feature) per group +/- SE, on a shared scale for both methods (DPO's +/-1 pair; GRPO's standardized advantages rescaled to match). 0 = the update is indifferent to that feature. Uses EVERY group (no embedding, no sampling).")
fig = pref.plot_lexical_push(LEX, palette=S.PALETTE)
if fig: eda_analysis.save_fig(fig, "update_lexical_push", caption="What each iteration's update pushes toward, both methods on one scale: completion length, question marks, affirmation and over-praise markers (+/-1 SE over groups)."); plt.show()

# ── 3b · the semantic half + the probe audit ──────────────────────────────────────
EMB = pref.embed_candidates(pref.sample_groups(CANDS))
DIRS = pref.direction_by_iter(EMB)
QUAL = pref.direction_quality(EMB, DIRS)
print("[probe audit] wins_holdout is the honest win rate; split_half_cos < ~0.5 means the "
      "per-iteration direction is not yet measured, whatever the projections show:")
display(QUAL.round(3))
eda_analysis.save_table(QUAL.round(4), "update_direction_quality", caption="Per (arm, iteration) probe audit of the update direction: in-sample wins_correct vs the honest wins_holdout (each half scored by the other half's direction), mean projection gap, and split_half_cos = cosine between directions fitted on disjoint halves (the precision check). A low split_half_cos means the direction is real but unmeasured at this group count, NOT that the update has no target.")

DIRS_ARM = pref.direction_by_arm(EMB)
QUAL_POOLED = pref.pooled_direction_quality(EMB, DIRS_ARM)
display(QUAL_POOLED.round(3))
eda_analysis.save_table(QUAL_POOLED.round(4), "update_direction_quality_pooled", caption="The same audit for the per-ARM direction pooled over all iterations — the estimate cross-method claims should rest on, since pooling multiplies the group count and the split-half cosine rises accordingly.")

COS = pref.pooled_direction_cosines(DIRS_ARM, QUAL_POOLED)
print("[pooled direction cosines] read `cosine_corrected` — `cosine` alone is capped by how well "
      "each direction is estimated (`ceiling`):")
display(COS.round(3))
eda_analysis.save_table(COS.round(4), "update_direction_cosines", caption="Cosine between every pair of arms' POOLED update directions, with the attenuation ceiling sqrt(r_a*r_b) from each direction's Spearman-Brown-corrected split-half reliability and the corrected cosine. 1.0 corrected = the two updates pull toward the same language; the ceiling is why a raw cosine of 0.3 is not interpretable on its own.")

# ── 3c · MI concepts, both methods on one figure ──────────────────────────────────
CATS = {arm: pref.category_projection(d) for arm, d in DIRS.items()}
fig = pref.plot_category_compare(CATS, palette=S.PALETTE,
                                 title="MI-concept preference by arm — both methods, same probe")
if fig: eda_analysis.save_fig(fig, "update_category_by_arm", caption="Each MI-concept word group projected onto the per-iteration update direction, one line per arm (both methods). Per-iteration directions: read alongside update_direction_quality, whose split_half_cos says how much of each curve is estimation noise."); plt.show()

# ── 3d · sanity gate: does the candidate-derived PTO direction match pairs.csv? ────
for arm_label, r in RESULTS.items():
    AG = pref.direction_agreement_with_pairs(EMB, arm_label, r["DIRS"])
    if not AG.empty:
        print(f"[sanity] {arm_label}: cos(candidate-derived, pairs.csv-derived) per iter = "
              f"{dict(zip(AG.train_iter, AG.cosine.round(3)))}")
        print(f"         mean {AG.cosine.mean():.3f} — two independent logs of the same DPO update; "
              "well below 1 would be a data-integrity problem, not a plotting one.")

## 4 · Does the training signal predict the eval move?  `[TRAINING] → [EVAL]`

**Purpose.** The link this notebook was missing. Everything above describes what the update *wanted*; this asks whether it explains what the model *did*. Each iteration's features are joined to the persona-paired eval delta that same update produced — update in **train_iter n** → adapter `iteration_n` → conversations `model_iter_n`, so its effect is `eval(model_iter_n) − eval(model_iter_{n-1})`, paired over the 96 shared personas (never a difference of iteration means: the personas are reshuffled every iteration).

**Read `rho_partial_iter`, not `spearman_rho`.** Nearly every feature here rises monotonically over training, and the eval deltas trend too (gains taper, MICI grows) — so *any* monotone feature correlates with *any* monotone delta, and the raw ρ is confounded with iteration index almost by construction. The partial removes `train_iter` from both sides: what survives it is "the iterations that pushed harder moved further **relative to where they sat in training**".

> ⚠️ **Correlational, n ≤ 10 per arm, uncorrected across the feature × metric grid.** With ~4 features × several metrics × 2 arms, one or two rows at *p* < .05 are expected by chance — a *pattern* across related features within one arm is worth something, an isolated star is not. This can also never separate "the update pushed affirmation, which raised MICI" from "iterations where the policy was drifting anyway also had affirmation-heavy branches". It is a mechanism consistent with the outcome curves, not a cause of them.
>
> The eval side is the **primary oracle only** — this notebook is training-side and refuses `EDA_JUDGE` (§1's cell), and the training signal genuinely cannot be re-graded. The cross-judge robustness of the MICI outcome itself lives in `8_Measurement_Validity`.

In [ ]:
FEATS = pref.preference_features_by_iter(CANDS, directions=DIRS, quality=QUAL)
LINK = pref.link_to_outcomes(FEATS, S.SCORES, metrics=S.METRICS)
if LINK.empty:
    print("no iteration has both its training signal and both eval endpoints scored.")
else:
    key = ["arm", "train_iter", "metric", "n_groups", "w_affirm", "w_question", "w_len",
           "w_overpraise", "delta_mean", "dz", "p"]
    display(LINK[[c for c in key if c in LINK.columns]].round(4))
    eda_analysis.save_table(LINK.round(4), "pref_outcome_link", caption="Per (arm, train_iter, metric): every feature of that iteration's update next to the persona-paired eval delta it produced (model_iter_n vs model_iter_n-1; delta_mean/dz/p from compare_two_models, N=96). The training-signal -> eval-move join. Primary oracle only (training-side notebook).")

    CORR = pref.outcome_correlations(LINK)
    top = CORR.reindex(CORR.rho_partial_iter.abs().sort_values(ascending=False).index)
    print("=== strongest links AFTER partialling out train_iter (the column to read) ===")
    display(top.head(15).round(3))
    eda_analysis.save_table(CORR.round(4), "pref_outcome_correlations", caption="Spearman rho between each update feature and the eval delta it preceded, per arm and pooled per method. spearman_rho is the raw association; rho_feature_vs_iter shows how much the feature simply trends with training; rho_partial_iter is the one to read (train_iter partialled out of both sides). Descriptive: n <= 10 iterations per arm, no multiplicity correction.")

    for feat in ["w_affirm", "w_overpraise", "w_question"]:
        fig = pref.plot_pref_outcome(LINK, feature=feat, palette=S.PALETTE)
        if fig:
            eda_analysis.save_fig(fig, f"pref_outcome_{feat}", caption=f"Update feature `{feat}` against the persona-paired eval delta of the same iteration, one point per iteration (label = iteration), per arm. Dashed = per-arm least squares, corner text = raw Spearman rho; the partial-correlation version is in pref_outcome_correlations.")
            plt.show()

## 5 · How to read this notebook
- **Is the probe real?** Read **`wins_holdout`** (§3), not §1's `wins_correct` — the latter scores the direction on the same groups it was fitted on and is optimistic by ~0.13 at PTO's group counts.
- **Is the probe *measured*?** `split_half_cos` (§3). Below ~0.5 the direction is not pinned down at that group count, so per-iteration projections built on it are mostly noise however smooth the curve looks. This is why §1–§2's drift artifacts carry the header's caveat and why cross-arm claims use the **pooled** direction.
- **What / how it drifts:** the word ranking + drift heatmap + learn/unlearn + MI-concept read-out test whether **affirmation/achievement** language becomes more preferred while **questions/reflection** fade — the latent-space signature of the behaviour drift in `2_Questionnaire_Detail` (MITI section). The exact, assumption-free version of the same test is §3's **lexical push**, which needs no embedding and uses every group.
- **Do the two methods want the same thing?** §3's `update_direction_cosines`, `cosine_corrected` column — raw cosines are capped by how well each direction is estimated.
- **Did wanting it work?** §4. Read `rho_partial_iter`; treat everything there as descriptive.
- **Caveat:** §1–§3 are the **[TRAINING]** signal (what the loss optimises), not the eval. Whether a shift is *good* is an eval/behaviour question (`1_Outcomes` / `3_Validity_and_Hacking` / `7_Stats`); §4 is the only place the two sides meet, and it meets them correlationally.

In [ ]:
print("index ->", eda_analysis.build_index())